# Dataset preparation for nuScenes subset experiments

## Goal

Prepare the nuScenes dataset for MMDetection3D and create a reproducible reduced training subset for budget-constrained experiments.

## Rationale

The subset is created at the **scene level**, not by randomly dropping frames. This keeps temporal consistency and makes later model comparisons cleaner.

## Outputs

This notebook will:

1. check the raw nuScenes folder structure
2. check existing MMDetection3D info files
3. inspect train/validation scene coverage
4. create a fixed train-scene subset
5. generate reduced `.pkl` info files
6. save the subset definition for reproducibility

# Setup

In [1]:
import json
import random
import numpy as np
import pandas as pd
import pickle

from pathlib import Path
from typing import List, Tuple, Final, Dict
from IPython.display import display
from nuscenes.nuscenes import NuScenes

# Paths

The nuScenes dataset is stored outside the Git repository because the full dataset is extremely large and must be stored on the appropriate UBELIX storage systems.

On the UBELIX HPC cluster, the dataset locations are:

- mini dataset: `/storage/homefs/ae04q066/datasets/nuscenes_mini`
- full dataset: `/rs_scratch/users/ae04q066/nuscenes_full`

The project uses a repository-local `data/nuscenes` symbolic link because MMDetection3D expects the dataset to be available at a fixed project-relative location. Using a symbolic link allows the codebase and configuration files to consistently reference `data/nuscenes` while the actual dataset remains stored elsewhere on the HPC filesystem.

The mini dataset is stored in persistent home storage and is mainly used for debugging and rapid experimentation. The full dataset is stored in scratch storage because it is significantly larger and primarily used for training and large-scale experiments.

In [2]:
# Define the main project paths and dataset configuration.

PROJECT_ROOT: Final[Path] = Path.cwd().resolve().parent

DATA_ROOT: Final[Path] = PROJECT_ROOT / "data" / "nuscenes"
MMDET3D_ROOT: Final[Path] = PROJECT_ROOT / "external" / "mmdetection3d"

NUSCENES_VERSION: Final[str] = "v1.0-trainval"

SUBSET_DIR: Final[Path] = DATA_ROOT / "subsets"

TRAIN_INFO_PATH: Final[Path] = DATA_ROOT / "nuscenes_infos_train.pkl"
VAL_INFO_PATH: Final[Path] = DATA_ROOT / "nuscenes_infos_val.pkl"

SUBSET_RATIO: Final[float] = 0.4
SUBSET_LABEL: Final[str] = f"{int(SUBSET_RATIO * 100)}pct"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATA_ROOT}")
print(f"Resolved dataset path: {DATA_ROOT.resolve()}")
print(f"MMDetection3D root: {MMDET3D_ROOT}")
print(f"nuScenes version: {NUSCENES_VERSION}")
print(f"Subset ratio: {SUBSET_RATIO}")
print(f"Subset label: {SUBSET_LABEL}")

Project root: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
Dataset root: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes
Resolved dataset path: /rs_scratch/users/ae04q066/nuscenes_full
MMDetection3D root: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d
nuScenes version: v1.0-trainval
Subset ratio: 0.4
Subset label: 40pct


In [3]:
print(f"DATA_ROOT exists: {DATA_ROOT.exists()}")
print(f"Is symbolic link: {DATA_ROOT.is_symlink()}")

if DATA_ROOT.is_symlink():
    print(f"Points to: {DATA_ROOT.resolve()}")

DATA_ROOT exists: True
Is symbolic link: True
Points to: /rs_scratch/users/ae04q066/nuscenes_full


# Initialize nuScenes

In [4]:
# Initialize the nuScenes dataset API.

nusc: NuScenes = NuScenes(
    version=NUSCENES_VERSION,
    dataroot=str(DATA_ROOT),
    verbose=True,
)

print(f"Number of scenes: {len(nusc.scene):,}")
print(f"Number of samples: {len(nusc.sample):,}")

Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 31.800 seconds.
Reverse indexing ...
Done reverse indexing in 8.2 seconds.
Number of scenes: 850
Number of samples: 34,149


In [5]:
# Verify the expected nuScenes directory structure.

required_dirs: List[str] = [
    "maps",
    "samples",
    "sweeps",
    NUSCENES_VERSION,
]

for directory in required_dirs:
    path: Path = DATA_ROOT / directory
    print(f"{directory:<20} exists={path.exists()}")

maps                 exists=True
samples              exists=True
sweeps               exists=True
v1.0-trainval        exists=True


In [6]:
print(f"Total scenes: {len(nusc.scene):,}")
print(f"Total samples: {len(nusc.sample):,}")

first_scene: Dict[str, str] = nusc.scene[0]

print(f"First scene name: {first_scene['name']}")
print(f"First scene token: {first_scene['token']}")

Total scenes: 850
Total samples: 34,149
First scene name: scene-0001
First scene token: 73030fb67d3c46cfb5e590168088ae39


# Load MMDetection3D info files

In [7]:
# Load the MMDetection3D train and validation info files.

with open(TRAIN_INFO_PATH, "rb") as file:
    train_data: Dict = pickle.load(file)

with open(VAL_INFO_PATH, "rb") as file:
    val_data: Dict = pickle.load(file)

print(f"Train info path: {TRAIN_INFO_PATH}")
print(f"Validation info path: {VAL_INFO_PATH}")

print(f"Train data type: {type(train_data)}")
print(f"Validation data type: {type(val_data)}")

print(f"Train keys: {list(train_data.keys())}")
print(f"Validation keys: {list(val_data.keys())}")

Train info path: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes/nuscenes_infos_train.pkl
Validation info path: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes/nuscenes_infos_val.pkl
Train data type: <class 'dict'>
Validation data type: <class 'dict'>
Train keys: ['metainfo', 'data_list']
Validation keys: ['metainfo', 'data_list']


In [8]:
# Inspect the MMDetection3D sample structure.

first_sample: Dict = train_data["data_list"][0]

print(f"Number of train samples: {len(train_data['data_list']):,}")
print(f"Number of validation samples: {len(val_data['data_list']):,}")

print("\nFirst train sample keys:")
print(list(first_sample.keys()))

Number of train samples: 28,130
Number of validation samples: 6,019

First train sample keys:
['sample_idx', 'token', 'timestamp', 'ego2global', 'images', 'lidar_points', 'instances', 'cam_instances']


In [9]:
print(first_sample["sample_idx"])
print(first_sample["token"])

0
e93e98b63d3b40209056d129dc53ceee


Each scene stores only the token of its first sample. The `next` field is then used to traverse the sequence and access all samples belonging to that scene.

```text
scene
 └── first_sample_token
        ↓
     sample 1  --next--> sample 2 --next--> sample 3 ...
```



In [10]:
# Build a mapping from sample tokens to nuScenes scene names.

sample_token_to_scene: Dict[str, str] = {}

for scene in nusc.scene:
    scene_name: str = scene["name"]
    sample_token: str = scene["first_sample_token"]

    while sample_token:
        sample: Dict = nusc.get("sample", sample_token)

        sample_token_to_scene[sample_token] = scene_name

        sample_token = sample["next"]

print(f"Mapped samples: {len(sample_token_to_scene):,}")

list(sample_token_to_scene.items())[:5]

Mapped samples: 34,149


[('e93e98b63d3b40209056d129dc53ceee', 'scene-0001'),
 ('14d5adfe50bb4445bc3aa5fe607691a8', 'scene-0001'),
 ('ae4e0c3aa3f24c91aab599e8b54e9264', 'scene-0001'),
 ('8de7ec06e1ac48c689c4d24d6cc64fd7', 'scene-0001'),
 ('ba94cb79ebc74614bc2442185cb53c26', 'scene-0001')]

In [11]:
# Add the corresponding scene name to each MMDetection3D sample
# using the mapping built from the nuScenes API.

for sample in train_data["data_list"]:
    token: str = sample["token"]
    sample["scene_name"] = sample_token_to_scene[token]

print(f"Updated train samples: {len(train_data['data_list']):,}")

train_data["data_list"][0]

Updated train samples: 28,130


{'sample_idx': 0,
 'token': 'e93e98b63d3b40209056d129dc53ceee',
 'timestamp': 1531883530.449377,
 'ego2global': [[0.12388695031404495,
   -0.9922940135002136,
   -0.0021557167638093233,
   1010.1328125],
  [0.9920361042022705,
   0.12390392273664474,
   -0.022630713880062103,
   610.8111572265625],
  [0.022723423317074776, 0.0006651013973169029, 0.9997415542602539, 0.0],
  [0.0, 0.0, 0.0, 1.0]],
 'images': {'CAM_FRONT': {'img_path': 'n015-2018-07-18-11-07-57+0800__CAM_FRONT__1531883530412470.jpg',
   'cam2img': [[1266.417203046554, 0.0, 816.2670197447984],
    [0.0, 1266.417203046554, 491.50706579294757],
    [0.0, 0.0, 1.0]],
   'cam2ego': [[0.005684778559952974,
     -0.005636667832732201,
     0.9999679327011108,
     1.7007912397384644],
    [-0.9999834895133972,
     -0.0008371152798645198,
     0.0056801484897732735,
     0.01594563201069832],
    [0.000805071322247386,
     -0.9999837875366211,
     -0.005641333758831024,
     1.5109575986862183],
    [0.0, 0.0, 0.0, 1.0]],
   '

In [12]:
# Extract the unique training scene names from the MMDetection3D samples.

train_scene_names: List[str] = sorted({
    sample["scene_name"]
    for sample in train_data["data_list"]
})

print(f"Unique train scenes: {len(train_scene_names):,}")

train_scene_names[:10]

Unique train scenes: 700


['scene-0001',
 'scene-0002',
 'scene-0004',
 'scene-0005',
 'scene-0006',
 'scene-0007',
 'scene-0008',
 'scene-0009',
 'scene-0010',
 'scene-0011']

# Create training subset

In [13]:
# Select complete scenes rather than individual samples.
# This preserves the original nuScenes scene structure and avoids
# creating fragmented scenes in the reduced training set.
# The subset is selected deterministically so that the same
# subset definition is reproduced across notebook executions.

num_subset_scenes: int = int(len(train_scene_names) * SUBSET_RATIO)

selected_scene_names: List[str] = train_scene_names[:num_subset_scenes]

print(f"Total train scenes: {len(train_scene_names):,}")
print(f"Selected subset scenes: {len(selected_scene_names):,}")

selected_scene_names[:10]

Total train scenes: 700
Selected subset scenes: 280


['scene-0001',
 'scene-0002',
 'scene-0004',
 'scene-0005',
 'scene-0006',
 'scene-0007',
 'scene-0008',
 'scene-0009',
 'scene-0010',
 'scene-0011']

In [14]:
# Keep only samples belonging to the selected subset scenes.

subset_data_list: List[Dict] = [
    sample
    for sample in train_data["data_list"]
    if sample["scene_name"] in selected_scene_names
]

print(f"Subset samples: {len(subset_data_list):,}")

subset_data_list[0]

Subset samples: 11,139


{'sample_idx': 0,
 'token': 'e93e98b63d3b40209056d129dc53ceee',
 'timestamp': 1531883530.449377,
 'ego2global': [[0.12388695031404495,
   -0.9922940135002136,
   -0.0021557167638093233,
   1010.1328125],
  [0.9920361042022705,
   0.12390392273664474,
   -0.022630713880062103,
   610.8111572265625],
  [0.022723423317074776, 0.0006651013973169029, 0.9997415542602539, 0.0],
  [0.0, 0.0, 0.0, 1.0]],
 'images': {'CAM_FRONT': {'img_path': 'n015-2018-07-18-11-07-57+0800__CAM_FRONT__1531883530412470.jpg',
   'cam2img': [[1266.417203046554, 0.0, 816.2670197447984],
    [0.0, 1266.417203046554, 491.50706579294757],
    [0.0, 0.0, 1.0]],
   'cam2ego': [[0.005684778559952974,
     -0.005636667832732201,
     0.9999679327011108,
     1.7007912397384644],
    [-0.9999834895133972,
     -0.0008371152798645198,
     0.0056801484897732735,
     0.01594563201069832],
    [0.000805071322247386,
     -0.9999837875366211,
     -0.005641333758831024,
     1.5109575986862183],
    [0.0, 0.0, 0.0, 1.0]],
   '

In [15]:
subset_train_data: Dict = train_data.copy()
subset_train_data["data_list"] = subset_data_list

print(f"Original train samples: {len(train_data['data_list']):,}")
print(f"Subset train samples: {len(subset_train_data['data_list']):,}")

print(f"Subset keys: {list(subset_train_data.keys())}")

Original train samples: 28,130
Subset train samples: 11,139
Subset keys: ['metainfo', 'data_list']


In [16]:
# Compare the original and reduced MMDetection3D dataset structures.

first_train_sample: Dict = train_data["data_list"][0]
first_subset_sample: Dict = subset_train_data["data_list"][0]

print(f"Original train samples: {len(train_data['data_list']):,}")
print(f"Validation samples: {len(val_data['data_list']):,}")
print(f"Subset train samples: {len(subset_train_data['data_list']):,}")

print("\nOriginal train sample keys:")
print(list(first_train_sample.keys()))

print("\nSubset train sample keys:")
print(list(first_subset_sample.keys()))

Original train samples: 28,130
Validation samples: 6,019
Subset train samples: 11,139

Original train sample keys:
['sample_idx', 'token', 'timestamp', 'ego2global', 'images', 'lidar_points', 'instances', 'cam_instances', 'scene_name']

Subset train sample keys:
['sample_idx', 'token', 'timestamp', 'ego2global', 'images', 'lidar_points', 'instances', 'cam_instances', 'scene_name']


# Save subset files

In [17]:
# Save the subset definition to preserve the exact scene selection used for the reduced training dataset.

SUBSET_SCENES_PATH: Final[Path] = (
    SUBSET_DIR / f"train_scene_subset_{SUBSET_LABEL}.json"
)

subset_definition: Dict = {
    "subset_fraction": SUBSET_RATIO,
    "num_train_scenes_total": len(train_scene_names),
    "num_train_scenes_subset": len(selected_scene_names),
    "selected_train_scene_names": selected_scene_names,
}

SUBSET_DIR.mkdir(parents=True, exist_ok=True)

if not SUBSET_SCENES_PATH.exists():
    with open(SUBSET_SCENES_PATH, "w") as file:
        json.dump(subset_definition, file, indent=2)

    print(f"Saved subset definition: {SUBSET_SCENES_PATH}")

else:
    print(f"File already exists: {SUBSET_SCENES_PATH}")

print(f"Subset scenes: {len(selected_scene_names):,}")

subset_definition.keys()

Saved subset definition: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes/subsets/train_scene_subset_40pct.json
Subset scenes: 280


dict_keys(['subset_fraction', 'num_train_scenes_total', 'num_train_scenes_subset', 'selected_train_scene_names'])

In [18]:
# Save the reduced MMDetection3D training info file
# corresponding to the selected scene subset.

SUBSET_TRAIN_INFO_PATH: Final[Path] = (
    SUBSET_DIR / f"nuscenes_infos_train_{SUBSET_LABEL}.pkl"
)

if not SUBSET_TRAIN_INFO_PATH.exists():
    with open(SUBSET_TRAIN_INFO_PATH, "wb") as file:
        pickle.dump(subset_train_data, file)

    print(f"Saved subset train info: {SUBSET_TRAIN_INFO_PATH}")

else:
    print(f"File already exists: {SUBSET_TRAIN_INFO_PATH}")

print(
    f"{SUBSET_LABEL} subset samples: "
    f"{len(subset_train_data['data_list']):,}"
)

Saved subset train info: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes/subsets/nuscenes_infos_train_40pct.pkl
40pct subset samples: 11,139


# Load subset files

In [19]:
# Reload the saved subset info file to verify that
# the serialized MMDetection3D dataset is valid.

with open(SUBSET_TRAIN_INFO_PATH, "rb") as file:
    loaded_subset_train_data: Dict = pickle.load(file)

print(
    f"Reloaded subset samples: "
    f"{len(loaded_subset_train_data['data_list']):,}"
)

loaded_subset_train_data.keys()

Reloaded subset samples: 11,139


dict_keys(['metainfo', 'data_list'])